In [1]:
import json
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt
import pickle as pl
from sklearn.svm import SVC
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import StratifiedKFold, train_test_split, cross_validate
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.neural_network import MLPClassifier
from sklearn.metrics import (confusion_matrix, classification_report, roc_auc_score, roc_curve, accuracy_score)
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats
import math
from scipy.stats import ttest_ind

In [ ]:
# '/home/aditya/hack_model/ag_news_1_0.01_linear_Adam', '/home/aditya/hack_model/ag_news_1_0.02_linear_Adam'

In [11]:
output_dir = './ag_news_6e-06_[0.35, 0.05, 0.3, 0.3]'

In [12]:
## my input features
alignment_matrix = np.load(f"{output_dir}/alignment_matrix_M.npy")

FileNotFoundError: [Errno 2] No such file or directory: './ag_news_6e-06_[0.35, 0.05, 0.3, 0.3]/alignment_matrix_M.npy'

In [13]:
with open(f'{output_dir}/accuracy_arr.pkl','rb') as f:
    accuracy_arr = pl.load(f)

FileNotFoundError: [Errno 2] No such file or directory: './ag_news_6e-06_[0.35, 0.05, 0.3, 0.3]/accuracy_arr.pkl'

In [14]:
with open(f'{output_dir}/dataset_info.json') as f:
    dataset_info =json.load(f)

FileNotFoundError: [Errno 2] No such file or directory: './ag_news_6e-06_[0.35, 0.05, 0.3, 0.3]/dataset_info.json'

In [15]:
labels = dataset_info['dataset']

NameError: name 'dataset_info' is not defined

In [16]:
sns.heatmap(alignment_matrix)
plt.show()

NameError: name 'alignment_matrix' is not defined

In [17]:
with open(f'{output_dir}/predictions.json', 'r') as file:
    data = json.load(file)

FileNotFoundError: [Errno 2] No such file or directory: './ag_news_6e-06_[0.35, 0.05, 0.3, 0.3]/predictions.json'

In [18]:
with open(f'{output_dir}.json') as f:
    config = json.load(f)

FileNotFoundError: [Errno 2] No such file or directory: './ag_news_6e-06_[0.35, 0.05, 0.3, 0.3].json'

In [19]:
ground_truth = np.array(data["indicator_vector_true"])

NameError: name 'data' is not defined

In [20]:
# For each pseudo-expert 
# The distribution of the alignment scores (for true point and non true points)
# The average +-std alignment score  (for true points and non true points)
# No of projections >0  and projections <0 (for true and non true points)
# No of projections <0 and projections<0 (for  true and non true points)

In [21]:
# Overall picture of all datapoints 

# No of points and their corresponding projections distribution (projections >0 is capped by the no of pseudo experts)
#(for true point and non true point)- make it general function for projection>epsilon (I guess)

# No of points and their corresponding projections distribution (projections <0 is capped by the no of pseudo experts)
# (for true point and non true point) -  make it general function for projection>epsilon (I guess)

# No of points and their Average alignment score (for true point and non true point)

In [22]:
def plot_average_alignment(alignment_matrix,ground_truth):
    """
    Visualize the distribution of average alignment scores for members vs non-members.
    
    This function computes the average alignment score across all pseudo-experts for each sample,
    then creates visualizations comparing the distributions between training set members and non-members.
    
    Args:
        alignment_matrix (np.ndarray): Shape (n_samples, n_pseudo_experts) - alignment scores
        ground_truth (np.ndarray): Binary array where 1=member, 0=non-member
    
    Returns:
        None: Displays matplotlib plots
    """
    member_mask = ground_truth == 1
    non_member_mask = ground_truth == 0
    
    fig, axes = plt.subplots(1, 2, figsize=(15, 10))
        
    # Average alignment scores
    member_avg = np.mean(alignment_matrix[member_mask], axis=1)
    non_member_avg = np.mean(alignment_matrix[non_member_mask], axis=1)
    
    axes[0].hist(member_avg, bins=30, alpha=0.7, label='Members', density=True)
    axes[0].hist(non_member_avg, bins=30, alpha=0.7, label='Non-members', density=True)
    axes[0].set_title('Distribution of Average Alignment Scores')
    axes[0].set_xlabel('Average Alignment Score')
    axes[0].set_ylabel('Density')
    axes[0].legend()
    axes[0].grid(True, alpha=0.3)
    
    # Box plot comparison
    data_to_plot = [member_avg, non_member_avg]
    axes[1].boxplot(data_to_plot, labels=['Members', 'Non-members'])
    axes[1].set_title('Average Alignment Score Comparison')
    axes[1].set_ylabel('Average Alignment Score')
    axes[1].grid(True, alpha=0.3)
    
    plt.tight_layout()
    plt.show()

In [23]:
alignment_matrix.shape

NameError: name 'alignment_matrix' is not defined

In [24]:
def plot_projection_distributions(alignment_matrix, ground_truth, epsilon=0.0):
    """
    Plot distribution of positive/negative projections per sample
    
    Args:
        alignment_matrix: Shape (n_samples, n_pseudo_experts)
        ground_truth: Binary array
        epsilon: Threshold for positive/negative (default 0)
    """
    
    member_mask = ground_truth == 1
    non_member_mask = ground_truth == 0
    
    # Count projections > epsilon and < -epsilon for each sample
    pos_counts_members = np.sum(alignment_matrix[member_mask] > epsilon, axis=1)
    neg_counts_members = np.sum(alignment_matrix[member_mask] < epsilon, axis=1)
    
    pos_counts_non_members = np.sum(alignment_matrix[non_member_mask] > epsilon, axis=1)
    neg_counts_non_members = np.sum(alignment_matrix[non_member_mask] < epsilon, axis=1)
    
    fig, axes = plt.subplots(1, 2, figsize=(15, 10))
    
    # Positive projections
    axes[0].hist(pos_counts_members, bins=range(alignment_matrix.shape[1]+2), 
                   alpha=0.7, label='Members', density=True,align='mid')
    axes[0].hist(pos_counts_non_members, bins=range(alignment_matrix.shape[1]+2), 
                   alpha=0.7, label='Non-members', density=True,align='mid')
    axes[0].set_title(f'Distribution of Positive Projections (>{epsilon})')
    axes[0].set_xlabel('Number of Positive Projections')
    axes[0].set_ylabel('Density')
    axes[0].legend()
    axes[0].grid(True, alpha=0.3)
    # Set x-axis ticks explicitly
    axes[1].set_xticks(range(alignment_matrix.shape[1]+1))
    axes[1].set_xticklabels(range(alignment_matrix.shape[1]+1))
    
    # Negative projections
    axes[1].hist(neg_counts_members, bins=range(alignment_matrix.shape[1]+2), 
                   alpha=0.7, label='Members', density=True,align='mid')
    axes[1].hist(neg_counts_non_members, bins=range(alignment_matrix.shape[1]+2), 
                   alpha=0.7, label='Non-members', density=True,align='mid')
    axes[1].set_title(f'Distribution of Negative Projections (<{epsilon})')
    axes[1].set_xlabel('Number of Negative Projections')
    axes[1].set_ylabel('Density')
    axes[1].legend()
    axes[1].grid(True, alpha=0.3)
        # Set x-axis ticks explicitly
    axes[1].set_xticks(range(alignment_matrix.shape[1]+1))
    axes[1].set_xticklabels(range(alignment_matrix.shape[1]+1))
    
    
    # Print summary statistics
    print("\n" + "="*60)
    print(f"PROJECTION DISTRIBUTION SUMMARY (threshold = {epsilon})")
    print("="*60)
    
    print(f"Positive projections (>{epsilon}):")
    print(f"  Members     - Mean: {np.mean(pos_counts_members):5.2f} ± {np.std(pos_counts_members):4.2f}")
    print(f"  Non-members - Mean: {np.mean(pos_counts_non_members):5.2f} ± {np.std(pos_counts_non_members):4.2f}")
    
    print(f"\nNegative projections (<{epsilon}):")
    print(f"  Members     - Mean: {np.mean(neg_counts_members):5.2f} ± {np.std(neg_counts_members):4.2f}")
    print(f"  Non-members - Mean: {np.mean(neg_counts_non_members):5.2f} ± {np.std(neg_counts_non_members):4.2f}")


In [5]:
def analyze_alignment_matrix(alignment_matrix, ground_truth):
    """
    Comprehensive analysis of alignment matrix
    
    Args:
        alignment_matrix: Shape (n_samples, n_pseudo_experts)
        ground_truth: Binary array (1=member, 0=non-member)
    """
    
    # Separate data by membership
    member_mask = ground_truth == 1
    non_member_mask = ground_truth == 0
    
    member_scores = alignment_matrix[member_mask]
    non_member_scores = alignment_matrix[non_member_mask]
    
    n_pseudo_experts = alignment_matrix.shape[1]
    
    print("="*80)
    print("ALIGNMENT MATRIX ANALYSIS")
    print("="*80)
    print(f"Total samples: {len(ground_truth)}")
    print(f"Members: {np.sum(member_mask)} ({np.mean(member_mask)*100:.1f}%)")
    print(f"Non-members: {np.sum(non_member_mask)} ({np.mean(non_member_mask)*100:.1f}%)")
    print(f"Pseudo-experts: {n_pseudo_experts}")
    
    # 1. PER PSEUDO-EXPERT ANALYSIS
    print("\n" + "="*60)
    print("PER PSEUDO-EXPERT ANALYSIS")
    print("="*60)
    
    for k in range(n_pseudo_experts):
        print(f"\nPseudo-Expert {k+1}:")
        print("-" * 30)
        
        member_k = member_scores[:, k]
        non_member_k = non_member_scores[:, k]
        
        # Distribution statistics
        print(f"Members     - Mean: {np.mean(member_k):6.3f} ± {np.std(member_k):5.3f}")
        print(f"Non-members - Mean: {np.mean(non_member_k):6.3f} ± {np.std(non_member_k):5.3f}")
        
        # Positive/negative projections
        member_pos = np.sum(member_k > 0)
        member_neg = np.sum(member_k < 0)
        non_member_pos = np.sum(non_member_k > 0)
        non_member_neg = np.sum(non_member_k < 0)
        
        print(f"Members     - Positive: {member_pos:3d}, Negative: {member_neg:3d}")
        print(f"Non-members - Positive: {non_member_pos:3d}, Negative: {non_member_neg:3d}")
    
    # 2. OVERALL ANALYSIS
    print("\n" + "="*60)
    print("OVERALL ANALYSIS")
    print("="*60)
    
    # Average alignment score per sample
    member_avg_scores = np.mean(member_scores, axis=1)
    non_member_avg_scores = np.mean(non_member_scores, axis=1)
    
    print(f"Average alignment scores:")
    print(f"Members     - Mean: {np.mean(member_avg_scores):6.3f} ± {np.std(member_avg_scores):5.3f}")
    print(f"Non-members - Mean: {np.mean(non_member_avg_scores):6.3f} ± {np.std(non_member_avg_scores):5.3f}")
    
    return {
        'member_scores': member_scores,
        'non_member_scores': non_member_scores,
        'member_avg_scores': member_avg_scores,
        'non_member_avg_scores': non_member_avg_scores
    }

In [6]:
def plot_heatmap_analysis(alignment_matrix, ground_truth):
    """
    Create detailed heatmap visualizations
    """
    # Sort by membership for better visualization
    sort_idx = np.argsort(ground_truth)
    sorted_matrix = alignment_matrix[sort_idx]
    sorted_labels = ground_truth[sort_idx]
    
    # Find the boundary between non-members and members
    boundary = np.sum(sorted_labels == 0)
    
    fig, axes = plt.subplots(1, 3, figsize=(20, 6))
    
    # Full heatmap
    im1 = axes[0].imshow(sorted_matrix, aspect='auto', cmap='RdBu_r', vmin=-1, vmax=1)
    axes[0].axhline(y=boundary-0.5, color='yellow', linewidth=2, label='Member/Non-member boundary')
    axes[0].set_title('Alignment Matrix (Sorted by Membership)')
    axes[0].set_xlabel('Pseudo-expert')
    axes[0].set_ylabel('Sample (0=Non-member, 1=Member)')
    plt.colorbar(im1, ax=axes[0])
    
    # Average by membership
    member_avg = np.mean(alignment_matrix[ground_truth == 1], axis=0)
    non_member_avg = np.mean(alignment_matrix[ground_truth == 0], axis=0)
    
    avg_matrix = np.array([non_member_avg, member_avg])
    im2 = axes[1].imshow(avg_matrix, aspect='auto', cmap='RdBu_r')
    axes[1].set_title('Average Alignment by Membership')
    axes[1].set_xlabel('Pseudo-expert')
    axes[1].set_yticks([0, 1])
    axes[1].set_yticklabels(['Non-members', 'Members'])
    plt.colorbar(im2, ax=axes[1])
    
    # Difference plot
    diff = member_avg - non_member_avg
    
    print(f"\nPseudo-expert differences (Members - Non-members):")
    for i, d in enumerate(diff):
        print(f"Expert {i+1}: {d:6.3f}")
        
    im3 = axes[2].imshow(diff.reshape(1, -1), aspect='auto', cmap='RdBu_r')
    axes[2].set_title('Difference (Members - Non-members)')
    axes[2].set_xlabel('Pseudo-expert')
    axes[2].set_yticks([0])
    axes[2].set_yticklabels(['Difference'])
    plt.colorbar(im3, ax=axes[2])
    
    plt.tight_layout()
    plt.show()

In [7]:
# Basic analysis
results = analyze_alignment_matrix(alignment_matrix, ground_truth)

NameError: name 'alignment_matrix' is not defined

In [8]:
# Heatmap analysis
plot_heatmap_analysis(alignment_matrix, ground_truth)

NameError: name 'alignment_matrix' is not defined

In [9]:
# Projection distributions for different epsilon values
epsilon_values = [0.2,0.4,0.6,0.7]
for eps in epsilon_values:
    print(f"\n{'='*60}")
    print(f"ANALYSIS WITH EPSILON = {eps}")
    print(f"{'='*60}")
    plot_projection_distributions(alignment_matrix, ground_truth, epsilon=eps)


ANALYSIS WITH EPSILON = 0.2


NameError: name 'alignment_matrix' is not defined

In [10]:
plot_average_alignment(alignment_matrix,ground_truth)

NameError: name 'alignment_matrix' is not defined

In [ ]:
# No of projections should be capped at numpseudoexperts + 1 
# see the histogram of the projection values

# The gradients measure the relative loss across that axis
# It could be that datapoints trained for model earlier may not give as much gradient later on
# Can we somehow capture this?

In [ ]:
def plot_single_pseudoexpert_distribution(alignment_matrix, ground_truth, expert_idx):
    """
    Plot distribution of alignment scores for a specific pseudo-expert
    
    Args:
        alignment_matrix: Shape (n_samples, n_pseudo_experts)
        ground_truth: Binary array (1=member, 0=non-member)
        expert_idx: Index of pseudo-expert to analyze (0-based)
    """
    
    member_mask = ground_truth == 1
    non_member_mask = ground_truth == 0
    
    # Get scores for this pseudo-expert
    member_scores = alignment_matrix[member_mask, expert_idx]
    non_member_scores = alignment_matrix[non_member_mask, expert_idx]
    
    fig, axes = plt.subplots(1, 3, figsize=(18, 5))
    
    # Histogram of alignment scores
    axes[0].hist(member_scores, bins=30, alpha=0.7, label='Members', density=True, color='blue')
    axes[0].hist(non_member_scores, bins=30, alpha=0.7, label='Non-members', density=True, color='red')
    axes[0].set_title(f'Pseudo-Expert {expert_idx+1}: Alignment Score Distribution')
    axes[0].set_xlabel('Alignment Score')
    axes[0].set_ylabel('Density')
    axes[0].legend()
    axes[0].grid(True, alpha=0.3)
    axes[0].axvline(x=0, color='black', linestyle='--', alpha=0.5, label='Zero line')
    
    # Box plot comparison
    data_to_plot = [member_scores, non_member_scores]
    axes[1].boxplot(data_to_plot, labels=['Members', 'Non-members'])
    axes[1].set_title(f'Pseudo-Expert {expert_idx+1}: Score Comparison')
    axes[1].set_ylabel('Alignment Score')
    axes[1].grid(True, alpha=0.3)
    axes[1].axhline(y=0, color='black', linestyle='--', alpha=0.5)
    
    # Violin plot
    axes[2].violinplot([member_scores, non_member_scores], positions=[1, 2])
    axes[2].set_xticks([1, 2])
    axes[2].set_xticklabels(['Members', 'Non-members'])
    axes[2].set_title(f'Pseudo-Expert {expert_idx+1}: Score Distribution Shape')
    axes[2].set_ylabel('Alignment Score')
    axes[2].grid(True, alpha=0.3)
    axes[2].axhline(y=0, color='black', linestyle='--', alpha=0.5)
    
    plt.tight_layout()
    plt.show()
    
    # Print statistics
    print(f"\nPseudo-Expert {expert_idx+1} Statistics:")
    print("="*50)
    print(f"Members     - Mean: {np.mean(member_scores):6.3f} ± {np.std(member_scores):5.3f}")
    print(f"              Range: [{np.min(member_scores):6.3f}, {np.max(member_scores):6.3f}]")
    print(f"              Positive: {np.sum(member_scores > 0):3d} / {len(member_scores):3d} ({np.mean(member_scores > 0)*100:.1f}%)")
    
    print(f"Non-members - Mean: {np.mean(non_member_scores):6.3f} ± {np.std(non_member_scores):5.3f}")
    print(f"              Range: [{np.min(non_member_scores):6.3f}, {np.max(non_member_scores):6.3f}]")
    print(f"              Positive: {np.sum(non_member_scores > 0):3d} / {len(non_member_scores):3d} ({np.mean(non_member_scores > 0)*100:.1f}%)")
    
    # Statistical test
    from scipy.stats import ttest_ind
    t_stat, p_value = ttest_ind(member_scores, non_member_scores)
    print(f"\nT-test: t={t_stat:.3f}, p={p_value:.6f}")
    print(f"Significant difference: {'Yes' if p_value < 0.05 else 'No'}")
    
for i in range(config['K']):
    plot_single_pseudoexpert_distribution(alignment_matrix, ground_truth, i)

## do it per class

In [25]:
def analyze_alignment_matrix_per_expert_per_class(alignment_matrix, class_labels):
    """
    Analyze alignment matrix per pseudo-expert, and within each pseudo-expert, per class.
    
    Args:
        alignment_matrix: np.ndarray, shape (n_samples, n_pseudo_experts)
        class_labels: 1D array-like of length n_samples with class labels
    """
    class_labels = np.array(class_labels)
    n_samples, n_pseudo_experts = alignment_matrix.shape
    unique_classes = np.unique(class_labels)
    
    print("=" * 80)
    print("ALIGNMENT MATRIX ANALYSIS (PER PSEUDO-EXPERT, PER CLASS)")
    print("=" * 80)
    print(f"Total samples: {n_samples}")
    print(f"Pseudo-experts: {n_pseudo_experts}")
    print(f"Num classes: {len(unique_classes)}")
    print("Class counts:")
    for c in unique_classes:
        cnt = np.sum(class_labels == c)
        print(f"  Class {c}: {cnt} samples ({cnt / n_samples * 100:.1f}%)")
    
    # Store programmatic results
    per_expert_results = {}
    
    print("\n" + "=" * 60)
    print("PER PSEUDO-EXPERT × CLASS ANALYSIS")
    print("=" * 60)
    
    for k in range(n_pseudo_experts):
        print(f"\nPseudo-Expert {k+1}:")
        print("-" * 40)
        
        # All scores for this pseudo-expert (one column)
        scores_k = alignment_matrix[:, k]  # shape (n_samples,)
        
        # Overall stats for this expert
        overall_mean = float(np.mean(scores_k))
        overall_std  = float(np.std(scores_k))
        overall_pos  = int(np.sum(scores_k > 0))
        overall_neg  = int(np.sum(scores_k < 0))
        
        print(f"  Overall:")
        print(f"    Mean: {overall_mean:6.3f} ± {overall_std:5.3f}")
        print(f"    Positive: {overall_pos:4d}, Negative: {overall_neg:4d}")
        
        # Per-class stats
        class_stats = {}
        print("\n  Per-class stats:")
        
        for c in unique_classes:
            class_mask = (class_labels == c)
            class_scores_k = scores_k[class_mask]  # all samples of class c for expert k
            
            if class_scores_k.size == 0:
                # Just in case there’s a class with zero samples
                print(f"    Class {c}: no samples")
                class_stats[c] = {
                    "mean": None,
                    "std": None,
                    "n": 0,
                    "n_pos": 0,
                    "n_neg": 0,
                }
                continue
            
            mean_c = float(np.mean(class_scores_k))
            std_c  = float(np.std(class_scores_k))
            n_c    = int(class_scores_k.size)
            pos_c  = int(np.sum(class_scores_k > 0))
            neg_c  = int(np.sum(class_scores_k < 0))
            
            print(f"    Class {c:>8} | n={n_c:4d} | "
                  f"Mean: {mean_c:6.3f} ± {std_c:5.3f} | "
                  f"Positive: {pos_c:4d}, Negative: {neg_c:4d}")
            
            class_stats[c] = {
                "mean": mean_c,
                "std": std_c,
                "n": n_c,
                "n_pos": pos_c,
                "n_neg": neg_c,
            }
        
        per_expert_results[k] = {
            "overall": {
                "mean": overall_mean,
                "std": overall_std,
                "n": int(scores_k.size),
                "n_pos": overall_pos,
                "n_neg": overall_neg,
            },
            "per_class": class_stats,
        }
    
    # return per_expert_results

In [26]:
analyze_alignment_matrix_per_expert_per_class(alignment_matrix, labels)

NameError: name 'alignment_matrix' is not defined

In [27]:
def plot_mean_projection_distribution_per_sample(
    alignment_matrix,
    ground_truth,
    class_labels,
    title="Mean projection distribution per sample (members vs non-members per class)"
):
    """
    Mean projection distribution per sample (members and non-members per class).
    
    Args:
        alignment_matrix: np.ndarray, shape (n_samples, n_pseudo_experts)
        ground_truth: 1D binary array-like, 1 = member, 0 = non-member
        class_labels: 1D array-like of length n_samples with class labels
        title: overall figure title
    """
    alignment_matrix = np.asarray(alignment_matrix)
    ground_truth = np.asarray(ground_truth)
    class_labels = np.asarray(class_labels)

    n_samples, n_pseudo_experts = alignment_matrix.shape
    unique_classes = np.unique(class_labels)

    # Mean projection per sample across pseudo-experts
    sample_means = alignment_matrix.mean(axis=1)

    # Masks for membership
    member_mask = (ground_truth == 1)
    non_member_mask = (ground_truth == 0)

    # Prepare figure layout: one subplot per class
    n_classes = len(unique_classes)
    n_cols = min(3, n_classes)
    n_rows = math.ceil(n_classes / n_cols)
    fig, axes = plt.subplots(n_rows, n_cols, figsize=(5 * n_cols, 4 * n_rows), squeeze=False)
    fig.suptitle(title, fontsize=14, y=1.02)

    # Collect stats to return
    stats = {}

    print("\n" + "=" * 80)
    print("MEAN PROJECTION DISTRIBUTION PER SAMPLE")
    print("Grouped by class and membership")
    print("=" * 80)

    for idx, cls in enumerate(unique_classes):
        r = idx // n_cols
        c = idx % n_cols
        ax = axes[r, c]

        class_mask = (class_labels == cls)

        # per-class & membership masks
        member_class_mask = member_mask & class_mask
        non_member_class_mask = non_member_mask & class_mask

        member_vals = sample_means[member_class_mask]
        non_member_vals = sample_means[non_member_class_mask]

        # Handle if one side is empty
        if member_vals.size > 0:
            ax.hist(member_vals, bins=30, alpha=0.6, density=True, label="Members")
        if non_member_vals.size > 0:
            ax.hist(non_member_vals, bins=30, alpha=0.6, density=True, label="Non-members")

        ax.set_title(f"Class {cls} (n={class_mask.sum()})")
        ax.set_xlabel("Mean projection per sample")
        ax.set_ylabel("Density")
        ax.grid(True, alpha=0.3)
        ax.legend()

        # Compute summary stats
        cls_stats = {}

        print(f"\nClass {cls} (total n={class_mask.sum()}):")
        print("-" * 40)

        if member_vals.size > 0:
            m_mean = float(np.mean(member_vals))
            m_std = float(np.std(member_vals))
            print(f"  Members     n={member_vals.size:4d} | "
                  f"Mean: {m_mean:7.4f} ± {m_std:6.4f}")
            cls_stats["members"] = {
                "n": int(member_vals.size),
                "mean": m_mean,
                "std": m_std,
            }
            
        else:
            print("  Members     n=   0 | (no samples)")
            cls_stats["members"] = {
                "n": 0,
                "mean": None,
                "std": None,
            }

        if non_member_vals.size > 0:
            nm_mean = float(np.mean(non_member_vals))
            nm_std = float(np.std(non_member_vals))
            print(f"  Non-members n={non_member_vals.size:4d} | "
                  f"Mean: {nm_mean:7.4f} ± {nm_std:6.4f}")
            cls_stats["non_members"] = {
                "n": int(non_member_vals.size),
                "mean": nm_mean,
                "std": nm_std,
            }
            
        else:
            print("  Non-members n=   0 | (no samples)")
            cls_stats["non_members"] = {
                "n": 0,
                "mean": None,
                "std": None,
            }

        stats[cls] = cls_stats

    # Hide unused subplots if any
    for idx in range(n_classes, n_rows * n_cols):
        r = idx // n_cols
        c = idx % n_cols
        fig.delaxes(axes[r, c])

    plt.tight_layout()
    # return {
    #     "sample_means": sample_means,
    #     "per_class_membership_stats": stats,
    # }

In [28]:
plot_mean_projection_distribution_per_sample(alignment_matrix,ground_truth,labels)

NameError: name 'alignment_matrix' is not defined

In [29]:
def plot_heatmap_analysis_per_class(alignment_matrix, ground_truth, class_labels):
    """
    Heatmap analysis per pseudo-expert, per class (members vs non-members).
    
    Args:
        alignment_matrix: np.ndarray, shape (n_samples, n_pseudo_experts)
        ground_truth: 1D binary array-like (1 = member, 0 = non-member)
        class_labels: 1D array-like, same length as ground_truth
    """
    alignment_matrix = np.asarray(alignment_matrix)
    ground_truth = np.asarray(ground_truth)
    class_labels = np.asarray(class_labels)

    n_samples, n_pseudo_experts = alignment_matrix.shape
    unique_classes = np.unique(class_labels)
    n_classes = len(unique_classes)

    # --- 1. Compute class-wise averages for members and non-members ---
    member_avgs = np.full((n_classes, n_pseudo_experts), np.nan)
    non_member_avgs = np.full((n_classes, n_pseudo_experts), np.nan)

    for i, cls in enumerate(unique_classes):
        cls_mask = (class_labels == cls)
        member_mask = (ground_truth == 1) & cls_mask
        non_member_mask = (ground_truth == 0) & cls_mask

        if np.any(non_member_mask):
            non_member_avgs[i] = alignment_matrix[non_member_mask].mean(axis=0)
        if np.any(member_mask):
            member_avgs[i] = alignment_matrix[member_mask].mean(axis=0)

    diff_avgs = member_avgs - non_member_avgs  # shape (n_classes, n_pseudo_experts)

    # --- 2. Plot heatmaps ---
    fig, axes = plt.subplots(1, 3, figsize=(22, 6))

    # Non-members
    im0 = axes[0].imshow(non_member_avgs, aspect='auto', cmap='RdBu_r')
    axes[0].set_title('Average Alignment by Class (Non-members)')
    axes[0].set_xlabel('Pseudo-expert')
    axes[0].set_ylabel('Class')
    axes[0].set_yticks(np.arange(n_classes))
    axes[0].set_yticklabels(unique_classes)
    plt.colorbar(im0, ax=axes[0])

    # Members
    im1 = axes[1].imshow(member_avgs, aspect='auto', cmap='RdBu_r')
    axes[1].set_title('Average Alignment by Class (Members)')
    axes[1].set_xlabel('Pseudo-expert')
    axes[1].set_ylabel('Class')
    axes[1].set_yticks(np.arange(n_classes))
    axes[1].set_yticklabels(unique_classes)
    plt.colorbar(im1, ax=axes[1])

    # Difference (Members - Non-members)
    im2 = axes[2].imshow(diff_avgs, aspect='auto', cmap='RdBu_r')
    axes[2].set_title('Difference (Members - Non-members) per Class')
    axes[2].set_xlabel('Pseudo-expert')
    axes[2].set_ylabel('Class')
    axes[2].set_yticks(np.arange(n_classes))
    axes[2].set_yticklabels(unique_classes)
    plt.colorbar(im2, ax=axes[2])

    plt.tight_layout()
    plt.show()

    # --- 3. Print summary differences ---
    print("\nPer-class pseudo-expert differences (Members - Non-members):")
    for i, cls in enumerate(unique_classes):
        print(f"\nClass {cls}:")
        for j, d in enumerate(diff_avgs[i]):
            print(f"  Expert {j+1}: {d:7.3f}")

    # return {
    #     "classes": unique_classes,
    #     "non_member_avgs": non_member_avgs,
    #     "member_avgs": member_avgs,
    #     "diff_avgs": diff_avgs,
    # }

In [30]:
plot_heatmap_analysis_per_class(alignment_matrix, ground_truth, labels)

NameError: name 'alignment_matrix' is not defined

In [31]:
def plot_single_pseudoexpert_distribution_per_class(alignment_matrix,ground_truth,class_labels,expert_idx):
    """
    For a specific pseudo-expert, plot score distributions per class,
    split into members vs non-members.
    
    Args:
        alignment_matrix: np.ndarray, shape (n_samples, n_pseudo_experts)
        ground_truth: 1D binary array-like (1=member, 0=non-member)
        class_labels: 1D array-like of length n_samples (class per sample)
        expert_idx: index of pseudo-expert to analyze (0-based)
    """
    alignment_matrix = np.asarray(alignment_matrix)
    ground_truth = np.asarray(ground_truth)
    class_labels = np.asarray(class_labels)

    n_samples, n_pseudo_experts = alignment_matrix.shape
    assert 0 <= expert_idx < n_pseudo_experts, "expert_idx out of range"

    # Scores for this pseudo-expert only
    expert_scores = alignment_matrix[:, expert_idx]

    unique_classes = np.unique(class_labels)
    n_classes = len(unique_classes)

    # Figure layout: one subplot per class
    n_cols = min(3, n_classes)
    n_rows = math.ceil(n_classes / n_cols)

    fig, axes = plt.subplots(n_rows, n_cols, figsize=(5 * n_cols, 4 * n_rows), squeeze=False)
    fig.suptitle(f"Pseudo-Expert {expert_idx+1}: Per-Class Score Distributions", fontsize=14, y=1.02)

    print(f"\nPseudo-Expert {expert_idx+1} per-class statistics")
    print("=" * 80)

    per_class_stats = {}

    for idx, cls in enumerate(unique_classes):
        r = idx // n_cols
        c = idx % n_cols
        ax = axes[r, c]

        cls_mask = (class_labels == cls)
        member_mask = (ground_truth == 1) & cls_mask
        non_member_mask = (ground_truth == 0) & cls_mask

        member_scores_cls = expert_scores[member_mask]
        non_member_scores_cls = expert_scores[non_member_mask]

        # Histogram
        if member_scores_cls.size > 0:
            ax.hist(member_scores_cls, bins=30, alpha=0.7, density=True, label='Members')
        if non_member_scores_cls.size > 0:
            ax.hist(non_member_scores_cls, bins=30, alpha=0.7, density=True, label='Non-members')

        ax.set_title(f"Class {cls} (n={cls_mask.sum()})")
        ax.set_xlabel("Alignment score")
        ax.set_ylabel("Density")
        ax.grid(True, alpha=0.3)
        ax.axvline(x=0, color='black', linestyle='--', alpha=0.5)
        ax.legend()

        # Stats & t-test per class
        cls_stats = {}

        print(f"\nClass {cls} (total n={cls_mask.sum()}):")
        print("-" * 60)

        # Members
        if member_scores_cls.size > 0:
            m_mean = float(np.mean(member_scores_cls))
            m_std = float(np.std(member_scores_cls))
            m_min = float(np.min(member_scores_cls))
            m_max = float(np.max(member_scores_cls))
            m_pos = int(np.sum(member_scores_cls > 0))
            print(f"  Members     n={member_scores_cls.size:4d} | "
                  f"Mean: {m_mean:7.3f} ± {m_std:6.3f} | "
                  f"Range: [{m_min:7.3f}, {m_max:7.3f}] | "
                  f"Positive: {m_pos:4d} ({m_pos/member_scores_cls.size*100:5.1f}%) | "
                  f"Negative: {member_scores_cls.size-m_pos:4d} ({(member_scores_cls.size-m_pos)/member_scores_cls.size*100:5.1f}%)")
        else:
            m_mean = m_std = m_min = m_max = None
            m_pos = 0
            print("  Members     n=   0 | (no samples)")

        cls_stats["members"] = {
            "n": int(member_scores_cls.size),
            "mean": m_mean,
            "std": m_std,
            "min": m_min,
            "max": m_max,
            "n_pos": m_pos,
            "values": member_scores_cls,
        }

        # Non-members
        if non_member_scores_cls.size > 0:
            nm_mean = float(np.mean(non_member_scores_cls))
            nm_std = float(np.std(non_member_scores_cls))
            nm_min = float(np.min(non_member_scores_cls))
            nm_max = float(np.max(non_member_scores_cls))
            nm_pos = int(np.sum(non_member_scores_cls > 0))
            print(f"  Non-members n={non_member_scores_cls.size:4d} | "
                  f"Mean: {nm_mean:7.3f} ± {nm_std:6.3f} | "
                  f"Range: [{nm_min:7.3f}, {nm_max:7.3f}] | "
                  f"Positive: {nm_pos:4d} ({nm_pos/non_member_scores_cls.size*100:5.1f}%) | "
                  f"Negative: {non_member_scores_cls.size-nm_pos:4d} ({(non_member_scores_cls.size-nm_pos)/non_member_scores_cls.size*100:5.1f}%)")
        else:
            nm_mean = nm_std = nm_min = nm_max = None
            nm_pos = 0
            print("  Non-members n=   0 | (no samples)")

        cls_stats["non_members"] = {
            "n": int(non_member_scores_cls.size),
            "mean": nm_mean,
            "std": nm_std,
            "min": nm_min,
            "max": nm_max,
            "n_pos": nm_pos,
            "values": non_member_scores_cls,
        }

        # T-test if both groups have >= 2 samples
        if member_scores_cls.size >= 2 and non_member_scores_cls.size >= 2:
            t_stat, p_value = ttest_ind(member_scores_cls, non_member_scores_cls, equal_var=False)
            print(f"  T-test (Members vs Non-members): t={t_stat:7.3f}, p={p_value:.6f} | "
                  f"Significant: {'Yes' if p_value < 0.05 else 'No'}")
            cls_stats["t_test"] = {
                "t": float(t_stat),
                "p": float(p_value),
                "significant_0.05": bool(p_value < 0.05),
            }
        else:
            print("  T-test: not computed (need ≥2 samples in each group)")
            cls_stats["t_test"] = None

        per_class_stats[cls] = cls_stats

    # Hide any unused subplots
    for idx in range(n_classes, n_rows * n_cols):
        r = idx // n_cols
        c = idx % n_cols
        fig.delaxes(axes[r, c])

    plt.tight_layout()
    plt.show()

    return per_class_stats

In [32]:
for i in range(config['K']):
    plot_single_pseudoexpert_distribution_per_class(alignment_matrix,ground_truth,labels,i)

NameError: name 'config' is not defined

In [33]:
def plot_class_distributions_across_pseudoexperts(alignment_matrix, ground_truth, class_labels):
    """
    For each class, plot member vs non-member score distributions
    across ALL pseudo-experts together.

    Args:
        alignment_matrix: np.ndarray, shape (n_samples, n_pseudo_experts)
        ground_truth: 1D binary array-like (1=member, 0=non-member)
        class_labels: 1D array-like of length n_samples (class per sample)
    """
    alignment_matrix = np.asarray(alignment_matrix)
    ground_truth = np.asarray(ground_truth)
    class_labels = np.asarray(class_labels)

    n_samples, n_pseudo_experts = alignment_matrix.shape
    unique_classes = np.unique(class_labels)

    all_stats = {}

    for cls in unique_classes:
        cls_mask = (class_labels == cls)
        member_mask = (ground_truth == 1) & cls_mask
        non_member_mask = (ground_truth == 0) & cls_mask

        member_scores_cls = alignment_matrix[member_mask]      # (n_members_c, n_pseudo_experts)
        non_member_scores_cls = alignment_matrix[non_member_mask]  # (n_non_members_c, n_pseudo_experts)

        # Prepare per-expert data for boxplots: list of arrays, one per expert
        member_data_per_expert = []
        non_member_data_per_expert = []

        for k in range(n_pseudo_experts):
            if member_scores_cls.size > 0:
                member_data_per_expert.append(member_scores_cls[:, k])
            else:
                member_data_per_expert.append(np.array([]))

            if non_member_scores_cls.size > 0:
                non_member_data_per_expert.append(non_member_scores_cls[:, k])
            else:
                non_member_data_per_expert.append(np.array([]))

        # Figure size scales a bit with number of experts
        fig, axes = plt.subplots(
            1, 2,
            figsize=(max(6, n_pseudo_experts * 0.7), 4),
            squeeze=False
        )
        ax_members = axes[0, 0]
        ax_non_members = axes[0, 1]

        fig.suptitle(f"Class {cls}: Distributions Across Pseudo-Experts", fontsize=14, y=1.02)

        # Boxplot for members
        if member_scores_cls.size > 0:
            ax_members.boxplot(
                member_data_per_expert,
                labels=[f"{i+1}" for i in range(n_pseudo_experts)],
                showfliers=False
            )
            ax_members.set_title(f"Members (n={member_scores_cls.shape[0]})")
        else:
            ax_members.text(0.5, 0.5, "No member samples", ha="center", va="center")
            ax_members.set_title("Members")
        ax_members.set_xlabel("Pseudo-expert")
        ax_members.set_ylabel("Alignment score")
        ax_members.axhline(y=0, color='black', linestyle='--', alpha=0.5)
        ax_members.grid(True, alpha=0.3)

        # Boxplot for non-members
        if non_member_scores_cls.size > 0:
            ax_non_members.boxplot(
                non_member_data_per_expert,
                labels=[f"{i+1}" for i in range(n_pseudo_experts)],
                showfliers=False
            )
            ax_non_members.set_title(f"Non-members (n={non_member_scores_cls.shape[0]})")
        else:
            ax_non_members.text(0.5, 0.5, "No non-member samples", ha="center", va="center")
            ax_non_members.set_title("Non-members")
        ax_non_members.set_xlabel("Pseudo-expert")
        ax_non_members.set_ylabel("Alignment score")
        ax_non_members.axhline(y=0, color='black', linestyle='--', alpha=0.5)
        ax_non_members.grid(True, alpha=0.3)

        plt.tight_layout()
        plt.show()

        # Optional: collect per-expert stats for this class
        class_stats = {"members": {}, "non_members": {}}
        for k in range(n_pseudo_experts):
            ms = member_data_per_expert[k]
            nms = non_member_data_per_expert[k]

            class_stats["members"][k] = {
                "n": int(ms.size),
                "mean": float(ms.mean()) if ms.size > 0 else None,
                "std": float(ms.std()) if ms.size > 0 else None,
            }
            class_stats["non_members"][k] = {
                "n": int(nms.size),
                "mean": float(nms.mean()) if nms.size > 0 else None,
                "std": float(nms.std()) if nms.size > 0 else None,
            }

        all_stats[cls] = class_stats

In [34]:
def plot_per_class_across_pseudoexperts(
    alignment_matrix,
    ground_truth,
    class_labels,
    bins=30
):
    """
    For each class, show distributions across ALL pseudo-experts:
    - Fix a class
    - For each pseudo-expert, plot member vs non-member score histograms
      in its own subplot.

    Args:
        alignment_matrix: (n_samples, n_pseudo_experts)
        ground_truth: 1D binary array-like (1 = member, 0 = non-member)
        class_labels: 1D array-like of length n_samples
        bins: number of histogram bins
    """
    alignment_matrix = np.asarray(alignment_matrix)
    ground_truth = np.asarray(ground_truth)
    class_labels = np.asarray(class_labels)

    n_samples, n_pseudo_experts = alignment_matrix.shape
    unique_classes = np.unique(class_labels)

    for cls in unique_classes:
        cls_mask = (class_labels == cls)
        member_mask = (ground_truth == 1) & cls_mask
        non_member_mask = (ground_truth == 0) & cls_mask

        member_scores_cls = alignment_matrix[member_mask]        # (n_members_c, n_experts)
        non_member_scores_cls = alignment_matrix[non_member_mask]  # (n_nonmembers_c, n_experts)

        print(f"\nClass {cls}: n_total={cls_mask.sum()}, "
              f"members={member_scores_cls.shape[0]}, "
              f"non-members={non_member_scores_cls.shape[0]}")

        # Layout: one axis per pseudo-expert
        n_cols = min(4, n_pseudo_experts)
        n_rows = math.ceil(n_pseudo_experts / n_cols)

        fig, axes = plt.subplots(
            n_rows, n_cols,
            figsize=(4 * n_cols, 3 * n_rows),
            squeeze=False
        )
        fig.suptitle(f"Class {cls}: Member vs Non-member across pseudo-experts", fontsize=14, y=1.02)

        for k in range(n_pseudo_experts):
            r = k // n_cols
            c = k % n_cols
            ax = axes[r, c]

            # Raw score distributions for this expert, this class
            m_scores = member_scores_cls[:, k] if member_scores_cls.size > 0 else np.array([])
            nm_scores = non_member_scores_cls[:, k] if non_member_scores_cls.size > 0 else np.array([])

            if m_scores.size > 0:
                ax.hist(m_scores, bins=bins, alpha=0.6, density=True, label="Members")
            if nm_scores.size > 0:
                ax.hist(nm_scores, bins=bins, alpha=0.6, density=True, label="Non-members")

            ax.set_title(f"Expert {k+1}")
            ax.set_xlabel("Alignment score")
            ax.set_ylabel("Density")
            ax.axvline(0, color="black", linestyle="--", alpha=0.5)
            ax.grid(True, alpha=0.3)
            ax.legend(fontsize=8)

        # Hide any unused axes
        for k in range(n_pseudo_experts, n_rows * n_cols):
            r = k // n_cols
            c = k % n_cols
            fig.delaxes(axes[r, c])

        plt.tight_layout()
        plt.show()

In [35]:
plot_per_class_across_pseudoexperts(alignment_matrix, ground_truth,labels)
plot_class_distributions_across_pseudoexperts(alignment_matrix, ground_truth, labels)

NameError: name 'alignment_matrix' is not defined

In [36]:
def plot_projection_distributions_per_class(alignment_matrix, ground_truth, class_labels, epsilon=0.0):
    """
    Plot distribution of positive/negative projections per sample, 
    split by class and membership.
    
    Args:
        alignment_matrix: Shape (n_samples, n_pseudo_experts)
        ground_truth: 1D binary array (1 = member, 0 = non-member)
        class_labels: 1D array-like of same length as ground_truth
        epsilon: Threshold for positive/negative (default 0)
    """
    alignment_matrix = np.asarray(alignment_matrix)
    ground_truth = np.asarray(ground_truth)
    class_labels = np.asarray(class_labels)

    n_samples, n_pseudo_experts = alignment_matrix.shape
    unique_classes = np.unique(class_labels)

    # Precompute counts per (class, membership)
    pos_counts = {}  # (cls, "member"/"non") -> array
    neg_counts = {}

    for cls in unique_classes:
        cls_mask = (class_labels == cls)
        member_mask = (ground_truth == 1) & cls_mask
        non_member_mask = (ground_truth == 0) & cls_mask

        # Count projections > epsilon and < -epsilon for each sample
        pos_counts_members = np.sum(alignment_matrix[member_mask] >  epsilon, axis=1)
        neg_counts_members = np.sum(alignment_matrix[member_mask] < -epsilon, axis=1)

        pos_counts_non_members = np.sum(alignment_matrix[non_member_mask] >  epsilon, axis=1)
        neg_counts_non_members = np.sum(alignment_matrix[non_member_mask] < -epsilon, axis=1)

        pos_counts[(cls, "member")] = pos_counts_members
        pos_counts[(cls, "non_member")] = pos_counts_non_members
        neg_counts[(cls, "member")] = neg_counts_members
        neg_counts[(cls, "non_member")] = neg_counts_non_members

    # Plot
    fig, axes = plt.subplots(1, 2, figsize=(15, 6))
    bins = range(n_pseudo_experts + 2)

    # Positive projections
    for cls in unique_classes:
        m_vals = pos_counts[(cls, "member")]
        nm_vals = pos_counts[(cls, "non_member")]

        if m_vals.size > 0:
            axes[0].hist(
                m_vals, bins=bins, alpha=0.5, density=True, align='mid',
                label=f'Class {cls} - Members'
            )
        if nm_vals.size > 0:
            axes[0].hist(
                nm_vals, bins=bins, alpha=0.5, density=True, align='mid',
                label=f'Class {cls} - Non-members'
            )

    axes[0].set_title(f'Distribution of Positive Projections (>{epsilon})')
    axes[0].set_xlabel('Number of Positive Projections')
    axes[0].set_ylabel('Density')
    axes[0].grid(True, alpha=0.3)
    axes[0].set_xticks(range(n_pseudo_experts + 1))
    axes[0].set_xticklabels(range(n_pseudo_experts + 1))
    axes[0].legend(fontsize=8)

    # Negative projections
    for cls in unique_classes:
        m_vals = neg_counts[(cls, "member")]
        nm_vals = neg_counts[(cls, "non_member")]

        if m_vals.size > 0:
            axes[1].hist(
                m_vals, bins=bins, alpha=0.5, density=True, align='mid',
                label=f'Class {cls} - Members'
            )
        if nm_vals.size > 0:
            axes[1].hist(
                nm_vals, bins=bins, alpha=0.5, density=True, align='mid',
                label=f'Class {cls} - Non-members'
            )

    axes[1].set_title(f'Distribution of Negative Projections (<{-epsilon})')
    axes[1].set_xlabel('Number of Negative Projections')
    axes[1].set_ylabel('Density')
    axes[1].grid(True, alpha=0.3)
    axes[1].set_xticks(range(n_pseudo_experts + 1))
    axes[1].set_xticklabels(range(n_pseudo_experts + 1))
    axes[1].legend(fontsize=8)

    plt.tight_layout()
    plt.show()

    # Print summary statistics
    print("\n" + "="*60)
    print(f"PROJECTION DISTRIBUTION SUMMARY BY CLASS (threshold = {epsilon})")
    print("="*60)

    print(f"\nPositive projections (>{epsilon}):")
    for cls in unique_classes:
        m_vals = pos_counts[(cls, "member")]
        nm_vals = pos_counts[(cls, "non_member")]

        print(f"  Class {cls}:")
        if m_vals.size > 0:
            print(f"    Members     n={len(m_vals):4d} | "
                  f"Mean: {np.mean(m_vals):5.2f} ± {np.std(m_vals):4.2f}")
        else:
            print("    Members     n=   0 | (no samples)")
        if nm_vals.size > 0:
            print(f"    Non-members n={len(nm_vals):4d} | "
                  f"Mean: {np.mean(nm_vals):5.2f} ± {np.std(nm_vals):4.2f}")
        else:
            print("    Non-members n=   0 | (no samples)")

    print(f"\nNegative projections (<{-epsilon}):")
    for cls in unique_classes:
        m_vals = neg_counts[(cls, "member")]
        nm_vals = neg_counts[(cls, "non_member")]

        print(f"  Class {cls}:")
        if m_vals.size > 0:
            print(f"    Members     n={len(m_vals):4d} | "
                  f"Mean: {np.mean(m_vals):5.2f} ± {np.std(m_vals):4.2f}")
        else:
            print("    Members     n=   0 | (no samples)")
        if nm_vals.size > 0:
            print(f"    Non-members n={len(nm_vals):4d} | "
                  f"Mean: {np.mean(nm_vals):5.2f} ± {np.std(nm_vals):4.2f}")
        else:
            print("    Non-members n=   0 | (no samples)")


In [37]:
epsilon_values = [0.2,0.4,0.6,0.7]
for eps in epsilon_values:
    print(f"\n{'='*60}")
    print(f"ANALYSIS WITH EPSILON = {eps}")
    print(f"{'='*60}")
    plot_projection_distributions_per_class(alignment_matrix, ground_truth, labels, epsilon=eps)


ANALYSIS WITH EPSILON = 0.2


NameError: name 'alignment_matrix' is not defined